In [4]:
import sys
import logging
import os
from pathlib import Path
from typing import Dict, Any
from instruments_service.cli.parser import parse_arguments
from instruments_service.cli.handlers import get_handler_for_mode

# Setup logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [6]:
import argparse

args = argparse.Namespace(
    mode="instruments",
    start_date="2023-05-23",
    end_date="2025-11-11",
    project_id="central-element-323112",
    gcs_bucket="market-data-tick",
    bigquery_dataset="market_data_hft",
    force=False,
    exchanges=None,
    CEFI=True,
    TRADFI=False,
    DEFI=False,
    query_type="list",
    venues=None,
    instrument_types=None,
    base_currency=None,
    quote_currency=None,
    symbol_pattern=None,
    instrument_id=None,
    instrument_ids=None,
    data_type=None,
    days_until_expiry=30,
    output_format="summary",
    output_file=None,
    limit=1000,
    log_level="INFO",
)

In [7]:
args.start_date

'2023-05-23'

In [ ]:
def main() -> Dict[str, Any]:
    """
    Main CLI entry point for instruments-service.

    Returns:
        Dictionary with operation results
    """
    try:
        # Parse arguments
        args = parse_arguments()

        # Setup logging level
        logging.getLogger().setLevel(getattr(logging, args.log_level.upper()))

        logger.info(f"🚀 Starting {args.mode} operation")
        if args.start_date:
            logger.info(
                f"📅 Date range: {args.start_date} to {args.end_date or args.start_date}"
            )

        # Build configuration from arguments
        config = {
            "project_id": args.project_id,
            "gcs_bucket": args.gcs_bucket,
            "bigquery_dataset": args.bigquery_dataset,
        }

        # Get handler for mode
        handler = get_handler_for_mode(args.mode, config)

        # Prepare arguments for handler
        handler_kwargs = {}

        # Date range
        if args.start_date:
            handler_kwargs["start_date"] = args.start_date
        if args.end_date:
            handler_kwargs["end_date"] = args.end_date

        # Common options
        if args.force:
            handler_kwargs["force"] = args.force
        if args.exchanges:
            handler_kwargs["exchanges"] = args.exchanges

        # Market type filters
        if args.CEFI:
            handler_kwargs["cefi"] = True
        if args.TRADFI:
            handler_kwargs["tradfi"] = True
        if args.DEFI:
            handler_kwargs["defi"] = True

        # Query-specific arguments
        if args.mode == "instruments-query":
            handler_kwargs["query_type"] = args.query_type
            if args.venues:
                handler_kwargs["venues"] = args.venues
            if args.instrument_types:
                handler_kwargs["instrument_types"] = args.instrument_types
            if args.base_currency:
                handler_kwargs["base_currency"] = args.base_currency
            if args.quote_currency:
                handler_kwargs["quote_currency"] = args.quote_currency
            if args.symbol_pattern:
                handler_kwargs["symbol_pattern"] = args.symbol_pattern
            if args.instrument_id:
                handler_kwargs["instrument_id"] = args.instrument_id
            if args.instrument_ids:
                handler_kwargs["instrument_ids"] = args.instrument_ids
            if args.data_type:
                handler_kwargs["data_type"] = args.data_type
            if args.days_until_expiry:
                handler_kwargs["days_until_expiry"] = args.days_until_expiry
            if args.output_format:
                handler_kwargs["output_format"] = args.output_format
            if args.output_file:
                handler_kwargs["output_file"] = args.output_file
            if args.limit:
                handler_kwargs["limit"] = args.limit

        # Execute handler
        result = handler.run(**handler_kwargs)

        # Cleanup
        handler.cleanup()

        # Log success
        if result.get("status") == "success" or result.get("success", True):
            logger.info(f"✅ {args.mode} operation completed successfully")
        else:
            logger.error(f"❌ {args.mode} operation failed")

        return result

    except Exception as e:
        logger.error(f"❌ CLI execution failed: {e}", exc_info=True)
        return {"success": False, "status": "error", "error": str(e)}


def run_cli():
    """Synchronous CLI execution"""
    try:
        result = main()
        return result
    except KeyboardInterrupt:
        logger.info("🛑 Operation cancelled by user")
        return {"success": False, "status": "error", "error": "Cancelled by user"}
    except Exception as e:
        logger.error(f"❌ Unexpected error: {e}", exc_info=True)
        return {"success": False, "status": "error", "error": str(e)}


result = run_cli()

2025-11-11 18:50:43,217 - instruments_service.app.core.cloud_instrument_storage - INFO - unified-cloud-services is available
usage: ipykernel_launcher.py [-h] --mode {instruments,instruments-query}
                             [--start-date START_DATE] [--end-date END_DATE]
                             [--project-id PROJECT_ID]
                             [--gcs-bucket GCS_BUCKET]
                             [--bigquery-dataset BIGQUERY_DATASET] [--force]
                             [--exchanges EXCHANGES [EXCHANGES ...]] [--CEFI]
                             [--TRADFI] [--DEFI]
                             [--query-type {list,summary,details,trading-params,data-types,expiring}]
                             [--venues VENUES [VENUES ...]]
                             [--instrument-types INSTRUMENT_TYPES [INSTRUMENT_TYPES ...]]
                             [--base-currency BASE_CURRENCY]
                             [--quote-currency QUOTE_CURRENCY]
                             [--sym

SystemExit: 2

/home/hk/.pyenv/versions/3.13.7/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
